# Vault Auto-Unseal with AWS KMS — On-Prem Vault accessing AWS KMS

This notebook demonstrates how to configure Vault's `seal "awskms"` when **Vault runs on-premises** (on a K8s cluster outside of AWS) and needs to access an **AWS KMS key** for auto-unseal.

### Why `role_arn` doesn't work in the seal stanza
`role_arn` is **not a supported parameter** in `seal "awskms"`. Vault silently ignores it and tries to authenticate with the base credentials.

### The solution: IAM user with KMS key policy
An IAM user is created in the AWS account with programmatic access (access key + secret key). The **KMS key policy** explicitly grants `kms:Encrypt`, `kms:Decrypt`, `kms:DescribeKey` to the IAM user. The credentials are injected into the Vault pod as environment variables — no STS, no profiles, no SDK config files.

### Architecture
```
┌─ On-Prem (K8s Cluster) ─────────┐      ┌─ AWS Account (KMS) ─────────────────┐
│                                  │      │                                      │
│  Vault Pod                       │      │  IAM User: vault-autounseal          │
│   ├─ AWS_ACCESS_KEY_ID (secret)  │      │   └─ IAM Policy:                     │
│   ├─ AWS_SECRET_ACCESS_KEY       │      │       kms:Encrypt/Decrypt/           │
│   └─ seal "awskms" { region,    │      │       DescribeKey on key ARN          │
│        kms_key_id } ─────────────┼──────┼──▶                                   │
│                                  │      │  KMS Key: vault-auto-unseal          │
│  Credentials injected via        │      │   └─ Key Policy grants:              │
│  extraSecretEnvironmentVars      │      │       kms:Encrypt/Decrypt/           │
│  from K8s secret                 │      │       DescribeKey                    │
│                                  │      │       to IAM user principal           │
└──────────────────────────────────┘      └──────────────────────────────────────┘
```

No AssumeRole. No AWS profiles. No SDK config files. Just IAM user credentials + KMS key policy.

## 1. Load base AWS credentials

In [ ]:
# Read aws credentials from csv file and set as environment variables
import csv, os
with open('vault_test_accessKeys.csv', 'r', encoding='utf-8-sig') as csvfile:
    reader = csv.DictReader(csvfile)
    creds = next(reader)
    access_key = creds['Access key ID']
    secret_key = creds['Secret access key']
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    print(f"Access Key: {access_key[:8]}...")
    print(f"Credentials loaded.")

## 2. Create KMS Key, IAM User, and Policies

Vault runs **on-prem** but needs access to an AWS KMS key for auto-unseal.  
We create:
- An **IAM user** in the AWS account with programmatic access (access key)
- A **KMS key** with a **key policy** that grants the IAM user direct access
- An **IAM policy** on the user that allows the KMS actions on the key ARN

Both sides (IAM policy + KMS key policy) must agree for access to work.

In [ ]:
import boto3
import json
import os
import time
from botocore.exceptions import ClientError

REGION = 'eu-west-3'
IAM_USER_NAME = 'vault-autounseal'
KMS_ALIAS = 'alias/vault-auto-unseal'
POLICY_NAME = 'vault-autounseal-kms-policy'

try:
    kms_client = boto3.client('kms', region_name=REGION)
    iam_client = boto3.client('iam')
    sts_client = boto3.client('sts')
    account_id = sts_client.get_caller_identity()['Account']

    # ─── 1. Create or reuse IAM User ───────────────────────────────────────
    try:
        iam_client.create_user(
            UserName=IAM_USER_NAME,
            Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal'}]
        )
        print(f"✓ IAM user '{IAM_USER_NAME}' created")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            print(f"✓ IAM user '{IAM_USER_NAME}' already exists — reusing")
            # Delete old access keys so we get fresh ones
            for key in iam_client.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
                iam_client.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
            print("  Old access keys deleted")
        else:
            raise

    ak_response = iam_client.create_access_key(UserName=IAM_USER_NAME)
    vault_access_key_id     = ak_response['AccessKey']['AccessKeyId']
    vault_secret_access_key = ak_response['AccessKey']['SecretAccessKey']
    print(f"✓ Access Key ID: {vault_access_key_id[:8]}...")

    # ─── 2. Create KMS Key ─────────────────────────────────────────────────
    # The key policy grants the IAM user direct access (required for on-prem Vault)
    kms_key_id = None
    try:
        existing = kms_client.describe_key(KeyId=KMS_ALIAS)
        kms_key_id  = existing['KeyMetadata']['KeyId']
        kms_key_arn = existing['KeyMetadata']['Arn']
        print(f"✓ KMS key already exists — reusing: {kms_key_id}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'NotFoundException':
            key_response = kms_client.create_key(
                Description='Vault Auto-Unseal Key (on-prem Vault)',
                KeyUsage='ENCRYPT_DECRYPT',
                Origin='AWS_KMS',
                Policy=json.dumps({
                    "Version": "2012-10-17",
                    "Id": "vault-auto-unseal-key-policy",
                    "Statement": [
                        {
                            "Sid": "EnableRootAccountFullAccess",
                            "Effect": "Allow",
                            "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
                            "Action": "kms:*",
                            "Resource": "*"
                        },
                        {
                            "Sid": "AllowVaultUnseal",
                            "Effect": "Allow",
                            "Principal": {"AWS": f"arn:aws:iam::{account_id}:user/{IAM_USER_NAME}"},
                            "Action": [
                                "kms:Encrypt",
                                "kms:Decrypt",
                                "kms:DescribeKey"
                            ],
                            "Resource": "*"
                        }
                    ]
                }),
                Tags=[{'TagKey': 'Purpose', 'TagValue': 'vault-auto-unseal'}]
            )
            kms_key_id  = key_response['KeyMetadata']['KeyId']
            kms_key_arn = key_response['KeyMetadata']['Arn']
            print(f"✓ KMS Key ID : {kms_key_id}")
            print(f"  KMS Key ARN: {kms_key_arn}")

            kms_client.create_alias(AliasName=KMS_ALIAS, TargetKeyId=kms_key_id)
            print(f"✓ KMS alias '{KMS_ALIAS}' created")
        else:
            raise

    # ─── 3. Attach IAM policy to the user ───────────────────────────────────
    kms_policy_document = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
            "Resource": kms_key_arn
        }]
    }

    kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    try:
        kms_policy_response = iam_client.create_policy(
            PolicyName=POLICY_NAME,
            Description='Allows KMS operations for Vault auto-unseal (on-prem Vault)',
            PolicyDocument=json.dumps(kms_policy_document)
        )
        kms_policy_arn = kms_policy_response['Policy']['Arn']
        print(f"✓ KMS policy created: {kms_policy_arn}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            print(f"✓ KMS policy '{POLICY_NAME}' already exists — reusing")
        else:
            raise

    try:
        iam_client.attach_user_policy(UserName=IAM_USER_NAME, PolicyArn=kms_policy_arn)
        print(f"✓ KMS policy attached to user")
    except ClientError:
        pass  # already attached

    # ─── 4. Persist values ──────────────────────────────────────────────────
    os.environ['KMS_KEY_ID']                  = kms_key_id
    os.environ['REGION']                      = REGION
    os.environ['VAULT_AWS_ACCESS_KEY_ID']     = vault_access_key_id
    os.environ['VAULT_AWS_SECRET_ACCESS_KEY'] = vault_secret_access_key

    print(f"\n✓ All values stored in environment")
    print(f"  KMS Key ARN: {kms_key_arn}")
    print(f"  IAM user credentials will be injected into Vault pod via K8s secret")

except ClientError as e:
    error_code = e.response['Error']['Code']
    error_msg  = e.response['Error']['Message']
    print(f"\n✗ AWS API error [{error_code}]: {error_msg}")
    print("  You may need to clean up partially created resources before retrying.")
    raise
except Exception as e:
    print(f"\n✗ Unexpected error: {e}")
    raise

## 3. Create K8S cluster

In [ ]:
! minikube delete --all

In [ ]:
! open -a Podman\ Desktop

In [ ]:
! minikube start -p workshop2 --force

In [ ]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

In [ ]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal 
%env K8S_CLUSTER_NAME=cluster.local 

In [ ]:
%%bash
rm -rf /tmp/vault
mkdir /tmp/vault

In [ ]:
! kubectl create namespace $VAULT_K8S_NAMESPACE

## 4. Create K8s Secret with AWS credentials

Store the IAM user's access key in a K8s secret. The Helm chart will inject these as environment variables via `extraSecretEnvironmentVars`. Since Vault runs on-prem (outside AWS), this is how we provide the credentials to reach the AWS KMS key. No AWS config files or SDK profiles needed.

In [ ]:
%%bash
# Create K8s secret with the IAM user credentials
kubectl delete secret vault-aws-creds --namespace "${VAULT_K8S_NAMESPACE}" --ignore-not-found
kubectl create secret generic vault-aws-creds \
  --namespace "${VAULT_K8S_NAMESPACE}" \
  --from-literal=AWS_ACCESS_KEY_ID="${VAULT_AWS_ACCESS_KEY_ID}" \
  --from-literal=AWS_SECRET_ACCESS_KEY="${VAULT_AWS_SECRET_ACCESS_KEY}"

echo "✓ Secret 'vault-aws-creds' created in namespace '${VAULT_K8S_NAMESPACE}'"

## 5. Generate TLS certificates

In [ ]:
%%bash

openssl genrsa -out ${WORKDIR}/vault.key 2048
cat > ${WORKDIR}/vault-csr.conf <<EOF
[req]
default_bits = 2048
prompt = no
encrypt_key = yes
default_md = sha256
distinguished_name = kubelet_serving
req_extensions = v3_req
[ kubelet_serving ]
O = system:nodes
CN = system:node:*.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
[ v3_req ]
basicConstraints = CA:FALSE
keyUsage = nonRepudiation, digitalSignature, keyEncipherment, dataEncipherment
extendedKeyUsage = serverAuth, clientAuth
subjectAltName = @alt_names
[alt_names]
DNS.1 = *.${VAULT_SERVICE_NAME}
DNS.2 = *.${VAULT_SERVICE_NAME}.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
DNS.3 = *.${VAULT_HELM_RELEASE_NAME}
DNS.4 = *.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
IP.1 = 127.0.0.1
EOF

openssl req -new -key ${WORKDIR}/vault.key -out ${WORKDIR}/vault.csr -config ${WORKDIR}/vault-csr.conf


cat > ${WORKDIR}/csr.yaml <<EOF
apiVersion: certificates.k8s.io/v1
kind: CertificateSigningRequest
metadata:
   name: vault.svc
spec:
   signerName: kubernetes.io/kubelet-serving
   expirationSeconds: 8640000
   request: $(cat ${WORKDIR}/vault.csr|base64|tr -d '\n')
   usages:
   - digital signature
   - key encipherment
   - server auth
EOF

kubectl create -f ${WORKDIR}/csr.yaml
kubectl certificate approve vault.svc
kubectl get csr vault.svc
kubectl get csr vault.svc -o jsonpath='{.status.certificate}' | openssl base64 -d -A -out ${WORKDIR}/vault.crt
kubectl config view \
--raw \
--minify \
--flatten \
-o jsonpath='{.clusters[].cluster.certificate-authority-data}' \
| base64 -d > ${WORKDIR}/vault.ca



kubectl create secret generic vault-ha-tls \
   -n $VAULT_K8S_NAMESPACE \
   --from-file=vault.key=${WORKDIR}/vault.key \
   --from-file=vault.crt=${WORKDIR}/vault.crt \
   --from-file=vault.ca=${WORKDIR}/vault.ca

In [ ]:
%%bash
secret=$(cat vault.hclic)
kubectl create secret generic vault-ent-license --from-literal="license=${secret}" -n $VAULT_K8S_NAMESPACE

## 6. Helm overrides — On-Prem Vault with AWS KMS auto-unseal

Key points:
- **`extraSecretEnvironmentVars`** injects `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` from the K8s secret
- **No** AWS config files, no profiles, no `AWS_SDK_LOAD_CONFIG`
- **Seal stanza**: only `region` + `kms_key_id`
- Vault on-prem reaches AWS KMS using the IAM user credentials injected as env vars
- The KMS key policy + IAM user policy authorize the access

In [ ]:
%%bash
cat > ${WORKDIR}/overrides.yaml <<EOF
global:
   enabled: true
   tlsDisable: false

csi:
   enabled: true
   image:
      repository: "docker.io/hashicorp/vault-csi-provider"

injector:
   enabled: true
   image:
      repository: docker.io/hashicorp/vault-k8s
   agentImage:
      repository: docker.io/hashicorp/vault

logLevel: "trace"

server:
   image:
      repository: docker.io/hashicorp/vault-enterprise
      tag: 1.20.4-ent
   enterpriseLicense:
      secretName: vault-ent-license

   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key

   # Inject AWS credentials from K8s secret as env vars
   # (required for on-prem Vault to reach AWS KMS)
   extraSecretEnvironmentVars:
      - envName: AWS_ACCESS_KEY_ID
        secretName: vault-aws-creds
        secretKey: AWS_ACCESS_KEY_ID
      - envName: AWS_SECRET_ACCESS_KEY
        secretName: vault-aws-creds
        secretKey: AWS_SECRET_ACCESS_KEY

   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls

   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true

   standalone:
      enabled: false
   affinity: ""
   ha:
      enabled: true
      replicas: 3
      raft:
         enabled: true
         setNodeId: true
         config: |
            ui = true
            listener "tcp" {
               tls_disable = 0
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }
            storage "raft" {
               path = "/vault/data"

               retry_join {
                  auto_join             = "provider=k8s namespace=vault label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "vault-0.vault-internal"
               }

            }
            # AWS KMS auto-unseal — on-prem Vault reaching AWS KMS
            # Credentials injected as env vars (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
            # IAM user has direct KMS access via key policy + IAM policy
            seal "awskms" {
               region     = "${REGION}"
               kms_key_id = "${KMS_KEY_ID}"
            }
            telemetry {
               disable_hostname = true
               prometheus_retention_time = "12h"
            }
            disable_mlock = true
            service_registration "kubernetes" {}

   ui:
      enabled: true
      serviceType: "LoadBalancer"
      serviceNodePort: null
      externalPort: 8200

EOF

echo "✓ overrides.yaml written to ${WORKDIR}/overrides.yaml"
cat ${WORKDIR}/overrides.yaml

## 7. Deploy Vault with Helm

In [ ]:
%%bash
helm install ${VAULT_HELM_RELEASE_NAME} hashicorp/vault \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --values ${WORKDIR}/overrides.yaml

echo ""
echo "✓ Helm release '${VAULT_HELM_RELEASE_NAME}' deployed"
echo "  Waiting for pods to be scheduled..."
kubectl get pods -n ${VAULT_K8S_NAMESPACE}

### Verify the AWS credentials are injected into the pod
Confirm the AWS env vars are set before initializing. These allow the on-prem Vault to reach AWS KMS.

In [ ]:
%%bash
# Wait for pod to be running first
kubectl wait pod vault-0 \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --for=jsonpath='{.status.phase}'=Running \
  --timeout=120s

echo "=== AWS environment variables ==="
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- env | grep -E 'AWS_'

echo ""
echo "=== Vault logs (last 20 lines) ==="
kubectl logs vault-0 -n ${VAULT_K8S_NAMESPACE} --tail=20

## 8. Initialize Vault

With AWS KMS auto-unseal, Vault only needs to be **initialized** once.  
The IAM user credentials (injected as env vars) allow the on-prem Vault pod to reach the AWS KMS key directly via the key policy.

In [ ]:
%%bash
# Give Vault a few seconds to finish starting its listener
sleep 5

echo "--- Initializing Vault (recovery-shares=1, recovery-threshold=1) ---"
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  vault operator init \
  -recovery-shares=1 \
  -recovery-threshold=1 \
  -format=json > ${WORKDIR}/vault-init.json

echo ""
echo "✓ Init output saved to ${WORKDIR}/vault-init.json"
cat ${WORKDIR}/vault-init.json

In [ ]:
import json, os, re

try:
    with open('/tmp/vault/vault-init.json') as f:
        raw = f.read()

    if not raw.strip():
        raise ValueError("vault-init.json is empty – Vault init may have failed. Check the init cell output.")

    match = re.search(r'\{', raw)
    if match:
        raw = raw[match.start():]

    init_data = json.loads(raw)

    root_token   = init_data['root_token']
    recovery_key = init_data['recovery_keys_b64'][0]

    os.environ['VAULT_TOKEN'] = root_token
    print(f"Root Token   : {root_token[:12]}...")
    print(f"Recovery Key : {recovery_key[:12]}...  (store securely!)")

except FileNotFoundError:
    print("✗ /tmp/vault/vault-init.json not found – run the Vault init cell first.")
except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Could not parse vault-init.json: {e}")
    print("  Re-run the Vault init cell and check its output for errors.")
except (KeyError, IndexError) as e:
    print(f"✗ Unexpected init JSON structure: {e}")
    raise

In [ ]:
%%bash
echo "=== vault-0 status ==="
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 \
  --  vault status -tls-skip-verify

echo ""
echo "=== Seal Type should show 'awskms' ==="
echo "=== All Vault pods ==="
kubectl get pods -n ${VAULT_K8S_NAMESPACE} -l app.kubernetes.io/name=vault

# Clean up

In [ ]:
%%bash
# 1 – Remove Vault Helm release and namespace
helm uninstall ${VAULT_HELM_RELEASE_NAME} --namespace ${VAULT_K8S_NAMESPACE} || true
kubectl delete namespace ${VAULT_K8S_NAMESPACE} --ignore-not-found
echo "✓ Helm release and namespace removed"

In [ ]:
# 2 – Remove AWS resources (IAM user, policy, KMS key)
import boto3, os
from botocore.exceptions import ClientError

REGION        = os.environ.get('REGION', 'eu-west-3')
IAM_USER_NAME = 'vault-autounseal'
POLICY_NAME   = 'vault-autounseal-kms-policy'
KMS_ALIAS     = 'alias/vault-auto-unseal'

iam = boto3.client('iam')
kms = boto3.client('kms', region_name=REGION)
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"

# Delete access keys for the user
try:
    for key in iam.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
        iam.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
    print("✓ Access keys deleted")
except ClientError as e:
    print(f"⚠ Access keys: {e.response['Error']['Message']}")

# Detach ALL attached policies from user (covers renamed/moved policies)
try:
    attached = iam.list_attached_user_policies(UserName=IAM_USER_NAME)['AttachedPolicies']
    for pol in attached:
        iam.detach_user_policy(UserName=IAM_USER_NAME, PolicyArn=pol['PolicyArn'])
        print(f"✓ Detached policy '{pol['PolicyName']}' from user")
    if not attached:
        print("  No attached policies found on user")
except ClientError as e:
    print(f"⚠ Detach policies: {e.response['Error']['Message']}")

# Delete user
try:
    iam.delete_user(UserName=IAM_USER_NAME)
    print(f"✓ IAM user '{IAM_USER_NAME}' deleted")
except ClientError as e:
    print(f"⚠ IAM user: {e.response['Error']['Message']}")

# Detach policy from ALL remaining entities (users, groups, roles) then delete
try:
    entities = iam.list_entities_for_policy(PolicyArn=kms_policy_arn)
    for u in entities.get('PolicyUsers', []):
        iam.detach_user_policy(UserName=u['UserName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from user '{u['UserName']}'")
    for g in entities.get('PolicyGroups', []):
        iam.detach_group_policy(GroupName=g['GroupName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from group '{g['GroupName']}'")
    for r in entities.get('PolicyRoles', []):
        iam.detach_role_policy(RoleName=r['RoleName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from role '{r['RoleName']}'")
    iam.delete_policy(PolicyArn=kms_policy_arn)
    print(f"✓ IAM policy '{POLICY_NAME}' deleted")
except ClientError as e:
    print(f"⚠ Policy: {e.response['Error']['Message']}")

# Schedule KMS key deletion
try:
    alias_info = kms.describe_key(KeyId=KMS_ALIAS)
    kms_key_id = alias_info['KeyMetadata']['KeyId']
    kms.delete_alias(AliasName=KMS_ALIAS)
    kms.schedule_key_deletion(KeyId=kms_key_id, PendingWindowInDays=7)
    print(f"✓ KMS key '{kms_key_id}' scheduled for deletion in 7 days")
except ClientError as e:
    print(f"⚠ KMS key: {e.response['Error']['Message']}")

print("\n✓ AWS cleanup complete")

In [ ]:
%%bash
# 3 – Delete minikube cluster and temp files
minikube delete -p workshop
rm -rf ${WORKDIR}
echo "✓ Minikube cluster 'workshop' deleted and temp files removed"